In [ ]:
def main(datasources, start_date, end_date):
    """
    v2 评分导向改造（Score=0.3A+0.7B，B=ElasticNet 增量稳定性）：

    核心判断：质量/价值/资金在全局因子池中高度拥挤 → L1 易压至 0 → B 项天花板低。
    公榜 0.97+ 更可能来自「平台预处理后仍有稳定增量」的稀缺微观结构信号。

    改造要点：
      1) 主信号改用 bar1m 独特构造（VAMP / 10档斜率 / 分时段压力漂移 /
         OBI 稳定性 / 振幅型 Amihud），而非模板五档压力均值；
      2) 财务质量仅作软门控（0.55~1.45），不做拥挤因子汤；
      3) 对 BARRA 风格 + factorlib 常见列（估值/资金/换手）二次正交，抬升 B；
      4) bar1m 按 14 天分批聚合，避免全年分钟数据 OOM。

    factor 越大越好。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    financial_table = datasources["financial"]

    FIN_LOOKBACK = 400
    fin_query_start = (
        pd.to_datetime(start_date) - pd.Timedelta(days=FIN_LOOKBACK)
    ).strftime("%Y-%m-%d %H:%M:%S")

    def fill_natural_dates(df, sd, ed):
        natural_dates = pd.date_range(start=sd, end=ed)
        df = df.set_index("date")

        def _fill(group_df):
            filled = group_df.reindex(natural_dates).ffill()
            filled["instrument"] = group_df.name
            return filled

        out = df.groupby("instrument", group_keys=False).apply(_fill)
        return (
            out.reset_index()
            .rename(columns={"index": "date"})
            .dropna(subset=["instrument"])
        )

    def _z(s: pd.Series) -> pd.Series:
        sd = s.std(ddof=0)
        if sd is None or not np.isfinite(sd) or sd < 1e-12:
            return pd.Series(0.0, index=s.index)
        return (s - s.mean()) / sd

    def _ols_resid_by_date(frame: pd.DataFrame, y_col: str, x_cols) -> pd.Series:
        pieces = []
        for _, g in frame.groupby("date", sort=False):
            y = g[y_col]
            X = g[list(x_cols)]
            valid = y.notna() & X.notna().all(axis=1)
            out = pd.Series(np.nan, index=g.index, dtype="float64")
            n = int(valid.sum())
            if n >= max(25, len(x_cols) + 8):
                yv = y.loc[valid].to_numpy(dtype=float)
                Xv = np.column_stack([np.ones(n), X.loc[valid].to_numpy(dtype=float)])
                try:
                    beta, *_ = np.linalg.lstsq(Xv, yv, rcond=None)
                    out.loc[valid] = yv - Xv @ beta
                except np.linalg.LinAlgError:
                    out.loc[valid] = yv - yv.mean()
            else:
                out.loc[valid] = y.loc[valid] - y.loc[valid].mean()
            pieces.append(out)
        return pd.concat(pieces)

    # ========== 1) 财务：仅用于质量门控 ==========
    fin_sql = f"""
    WITH cte_ttm AS (
        SELECT
            date, instrument,
            net_profit_to_parent_shareholders AS np_ttm,
            operating_revenue AS revenue_ttm,
            gross_profit AS gp_ttm,
            net_cffoa AS cfo_ttm
        FROM {financial_table}
        WHERE category = 'ttm' AND shift = 0
    ),
    cte_lf AS (
        SELECT
            date, instrument,
            total_equity_to_parent_shareholders AS equity_lf,
            total_assets AS assets_lf,
            total_liabilities AS liab_lf,
            total_current_assets AS ca_lf,
            total_current_liabilities AS cl_lf
        FROM {financial_table}
        WHERE category = 'lf' AND shift = 0
    )
    SELECT
        t.date,
        t.instrument,
        t.np_ttm / NULLIF(l.equity_lf, 0) AS roe_ttm,
        t.np_ttm / NULLIF(l.assets_lf, 0) AS roa_ttm,
        t.np_ttm / NULLIF(t.revenue_ttm, 0) AS net_margin,
        t.gp_ttm / NULLIF(t.revenue_ttm, 0) AS gross_margin,
        t.cfo_ttm / NULLIF(ABS(t.np_ttm), 0) AS cfo_quality,
        l.liab_lf / NULLIF(l.assets_lf, 0) AS leverage,
        l.ca_lf / NULLIF(l.cl_lf, 0) AS current_ratio
    FROM cte_ttm t
    PRUNE JOIN cte_lf l USING (date, instrument)
    """
    fin_df = dai.query(fin_sql, filters={"date": [fin_query_start, end_date]}).df()
    fin_df["date"] = pd.to_datetime(fin_df["date"])
    fin_df["instrument"] = fin_df["instrument"].astype(str)
    fin_daily = fill_natural_dates(fin_df, fin_query_start, end_date)

    # ========== 2) 独特微观结构（主 Alpha）==========
    # 实表仅有五档；volume/amount 为分钟 bar 量额。
    # 全年分钟 bar + LAG/双 CTE 易 OOM：按半月分批，单层 CTE，无窗口函数。
    def _date_chunks(sd, ed, days=14):
        cur = pd.Timestamp(sd)
        end_ts = pd.Timestamp(ed)
        step = pd.Timedelta(days=days)
        while cur <= end_ts:
            ce = min(cur + step - pd.Timedelta(seconds=1), end_ts)
            yield (
                cur.strftime("%Y-%m-%d %H:%M:%S"),
                ce.strftime("%Y-%m-%d %H:%M:%S"),
            )
            cur = ce + pd.Timedelta(seconds=1)

    bar_sql = f"""
    WITH m AS (
        SELECT
            date_trunc('day', date)::DATE::DATETIME AS day,
            date,
            instrument,
            close, high, low, pre_close,
            amount::DOUBLE AS amount,
            volume::DOUBLE AS volume,
            deal_number::DOUBLE AS deal_number,
            (hour(date) * 60 + minute(date)) AS mod,
            (ask_price1 - bid_price1)
                / NULLIF((ask_price1 + bid_price1) / 2.0, 0) AS rel_spread,
            (
                COALESCE(bid_volume1,0)::DOUBLE * 1.0
                + COALESCE(bid_volume2,0)::DOUBLE * EXP(-0.25)
                + COALESCE(bid_volume3,0)::DOUBLE * EXP(-0.50)
                + COALESCE(bid_volume4,0)::DOUBLE * EXP(-0.75)
                + COALESCE(bid_volume5,0)::DOUBLE * EXP(-1.00)
                - COALESCE(ask_volume1,0)::DOUBLE * 1.0
                - COALESCE(ask_volume2,0)::DOUBLE * EXP(-0.25)
                - COALESCE(ask_volume3,0)::DOUBLE * EXP(-0.50)
                - COALESCE(ask_volume4,0)::DOUBLE * EXP(-0.75)
                - COALESCE(ask_volume5,0)::DOUBLE * EXP(-1.00)
            ) / NULLIF(
                COALESCE(bid_volume1,0)::DOUBLE + COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_volume3,0)::DOUBLE + COALESCE(bid_volume4,0)::DOUBLE
                + COALESCE(bid_volume5,0)::DOUBLE
                + COALESCE(ask_volume1,0)::DOUBLE + COALESCE(ask_volume2,0)::DOUBLE
                + COALESCE(ask_volume3,0)::DOUBLE + COALESCE(ask_volume4,0)::DOUBLE
                + COALESCE(ask_volume5,0)::DOUBLE, 0
            ) AS obi5,
            (
                COALESCE(bid_volume1,0)::DOUBLE + COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_volume3,0)::DOUBLE
                - COALESCE(ask_volume1,0)::DOUBLE - COALESCE(ask_volume2,0)::DOUBLE
                - COALESCE(ask_volume3,0)::DOUBLE
            ) / NULLIF(
                COALESCE(bid_volume1,0)::DOUBLE + COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_volume3,0)::DOUBLE
                + COALESCE(ask_volume1,0)::DOUBLE + COALESCE(ask_volume2,0)::DOUBLE
                + COALESCE(ask_volume3,0)::DOUBLE, 0
            ) AS depth3,
            (
                COALESCE(bid_volume1,0)::DOUBLE * 1.0
                + COALESCE(bid_volume2,0)::DOUBLE * 2.0
                + COALESCE(bid_volume3,0)::DOUBLE * 3.0
                + COALESCE(bid_volume4,0)::DOUBLE * 4.0
                + COALESCE(bid_volume5,0)::DOUBLE * 5.0
            ) / NULLIF(
                COALESCE(bid_volume1,0)::DOUBLE + COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_volume3,0)::DOUBLE + COALESCE(bid_volume4,0)::DOUBLE
                + COALESCE(bid_volume5,0)::DOUBLE, 0
            )
            - (
                COALESCE(ask_volume1,0)::DOUBLE * 1.0
                + COALESCE(ask_volume2,0)::DOUBLE * 2.0
                + COALESCE(ask_volume3,0)::DOUBLE * 3.0
                + COALESCE(ask_volume4,0)::DOUBLE * 4.0
                + COALESCE(ask_volume5,0)::DOUBLE * 5.0
            ) / NULLIF(
                COALESCE(ask_volume1,0)::DOUBLE + COALESCE(ask_volume2,0)::DOUBLE
                + COALESCE(ask_volume3,0)::DOUBLE + COALESCE(ask_volume4,0)::DOUBLE
                + COALESCE(ask_volume5,0)::DOUBLE, 0
            ) AS slope_asym,
            (
                COALESCE(bid_price1,0)::DOUBLE * COALESCE(bid_volume1,0)::DOUBLE
                + COALESCE(bid_price2,0)::DOUBLE * COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_price3,0)::DOUBLE * COALESCE(bid_volume3,0)::DOUBLE
                + COALESCE(bid_price4,0)::DOUBLE * COALESCE(bid_volume4,0)::DOUBLE
                + COALESCE(bid_price5,0)::DOUBLE * COALESCE(bid_volume5,0)::DOUBLE
            ) / NULLIF(
                COALESCE(bid_volume1,0)::DOUBLE + COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_volume3,0)::DOUBLE + COALESCE(bid_volume4,0)::DOUBLE
                + COALESCE(bid_volume5,0)::DOUBLE, 0
            ) AS p_bid_eff,
            (
                COALESCE(ask_price1,0)::DOUBLE * COALESCE(ask_volume1,0)::DOUBLE
                + COALESCE(ask_price2,0)::DOUBLE * COALESCE(ask_volume2,0)::DOUBLE
                + COALESCE(ask_price3,0)::DOUBLE * COALESCE(ask_volume3,0)::DOUBLE
                + COALESCE(ask_price4,0)::DOUBLE * COALESCE(ask_volume4,0)::DOUBLE
                + COALESCE(ask_price5,0)::DOUBLE * COALESCE(ask_volume5,0)::DOUBLE
            ) / NULLIF(
                COALESCE(ask_volume1,0)::DOUBLE + COALESCE(ask_volume2,0)::DOUBLE
                + COALESCE(ask_volume3,0)::DOUBLE + COALESCE(ask_volume4,0)::DOUBLE
                + COALESCE(ask_volume5,0)::DOUBLE, 0
            ) AS p_ask_eff,
            (
                COALESCE(bid_volume1,0)::DOUBLE + COALESCE(bid_volume2,0)::DOUBLE
                + COALESCE(bid_volume3,0)::DOUBLE + COALESCE(bid_volume4,0)::DOUBLE
                + COALESCE(bid_volume5,0)::DOUBLE
            ) AS q_bid5,
            (
                COALESCE(ask_volume1,0)::DOUBLE + COALESCE(ask_volume2,0)::DOUBLE
                + COALESCE(ask_volume3,0)::DOUBLE + COALESCE(ask_volume4,0)::DOUBLE
                + COALESCE(ask_volume5,0)::DOUBLE
            ) AS q_ask5,
            (COALESCE(bid_num_orders1,0) - COALESCE(ask_num_orders1,0))::DOUBLE
                / NULLIF(COALESCE(bid_num_orders1,0) + COALESCE(ask_num_orders1,0), 0)
                AS order_imb1,
            (high - low) / NULLIF(amount::DOUBLE, 0) AS amihud_bar
        FROM {bar1m}
        WHERE ask_price1 > 0 AND bid_price1 > 0 AND close > 0
    )
    SELECT
        day AS date,
        instrument,
        LAST(close ORDER BY date)
            / NULLIF(
                LAST(
                    (p_bid_eff * q_ask5 + p_ask_eff * q_bid5)
                        / NULLIF(q_bid5 + q_ask5, 0)
                ORDER BY date), 0
            ) - 1 AS vamp_dev,
        AVG(CASE WHEN mod >= 870 THEN obi5 END)
            - AVG(CASE WHEN mod < 630 THEN obi5 END) AS pressure_drift,
        AVG(obi5) AS obi_mean,
        AVG(depth3) AS depth3_mean,
        AVG(slope_asym) AS slope_asym,
        LAST(close ORDER BY date)
            / NULLIF(SUM(amount) / NULLIF(SUM(volume), 0), 0)
            - 1 AS vwap_dev,
        LAST(close ORDER BY date)
            / NULLIF(
                (LAST(p_bid_eff ORDER BY date) + LAST(p_ask_eff ORDER BY date)) / 2.0, 0
            ) - 1 AS avgpx_dev,
        STDDEV_POP(obi5) AS obi_instability,
        AVG(rel_spread) AS rel_spread,
        AVG(amihud_bar) AS amihud,
        AVG((high - low) / NULLIF(close, 0)) AS avg_range,
        STDDEV_POP(volume) / NULLIF(AVG(volume), 0) AS vol_cv,
        SUM(deal_number) / NULLIF(SUM(volume), 0) AS fragmentation,
        AVG(ABS(obi5 - order_imb1)) AS spoof_gap,
        (MAX(high) - MIN(low)) / NULLIF(LAST(pre_close ORDER BY date), 0) AS day_range
    FROM m
    GROUP BY day, instrument
    """
    bar_parts = []
    for cs, ce in _date_chunks(start_date, end_date, days=14):
        part = dai.query(
            bar_sql, filters={"date": [cs, ce]}, compression=True
        ).df()
        if part is not None and len(part) > 0:
            bar_parts.append(part)
    if not bar_parts:
        raise ValueError("bar1m query returned empty result")
    bar_df = pd.concat(bar_parts, ignore_index=True)
    del bar_parts
    bar_df["date"] = pd.to_datetime(bar_df["date"])
    bar_df["instrument"] = bar_df["instrument"].astype(str)

    # ========== 3) 风格暴露（预处理对齐 + 控制变量）==========
    exp_sql = """
    SELECT
        date, instrument,
        SIZE, BETA, MOMENTUM, RESVOL, SIZENL,
        BTOP, LIQUIDTY, EARNYILD, GROWTH, LEVERAGE,
        industry_level1_code, float_market_cap
    FROM bigalpha_2026_exposure
    """
    exp_df = dai.query(exp_sql, filters={"date": [start_date, end_date]}).df()
    exp_df["date"] = pd.to_datetime(exp_df["date"])
    exp_df["instrument"] = exp_df["instrument"].astype(str)
    exp_df["industry_level1_code"] = (
        exp_df["industry_level1_code"].fillna("UNKNOWN").astype(str)
    )

    # ========== 4) factorlib：仅作正交控制，不进主信号 ==========
    lib_sql = """
    SELECT
        date, instrument,
        pe_ttm, pb, turn,
        netflow_amount_rate_main,
        reversal_5, volatility_5,
        roe_avg_ttm
    FROM bigalpha_2026_factorlib
    """
    lib_df = dai.query(lib_sql, filters={"date": [start_date, end_date]}).df()
    lib_df["date"] = pd.to_datetime(lib_df["date"])
    lib_df["instrument"] = lib_df["instrument"].astype(str)

    # ========== 5) 股票池 ==========
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)

    df = stk_pool.copy()
    df = pd.merge(df, fin_daily, on=["date", "instrument"], how="left")
    df = pd.merge(df, bar_df, on=["date", "instrument"], how="left")
    df = pd.merge(df, exp_df, on=["date", "instrument"], how="left")
    df = pd.merge(df, lib_df, on=["date", "instrument"], how="left")
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    num_cols = [
        "roe_ttm", "roa_ttm", "net_margin", "gross_margin", "cfo_quality",
        "leverage", "current_ratio",
        "vamp_dev", "pressure_drift", "obi_mean", "depth3_mean", "slope_asym",
        "vwap_dev", "avgpx_dev", "obi_instability", "rel_spread", "amihud",
        "avg_range", "vol_cv", "fragmentation", "spoof_gap", "day_range",
        "SIZE", "BETA", "MOMENTUM", "RESVOL", "SIZENL",
        "BTOP", "LIQUIDTY", "EARNYILD", "GROWTH", "LEVERAGE", "float_market_cap",
        "pe_ttm", "pb", "turn", "netflow_amount_rate_main",
        "reversal_5", "volatility_5", "roe_avg_ttm",
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ----- 质量门控（轻量，避免拥挤）-----
    df["fin_quality"] = (
        df["roe_ttm"].fillna(0)
        + df["roa_ttm"].fillna(0)
        + 0.7 * df["net_margin"].fillna(0)
        + 0.4 * df["gross_margin"].fillna(0)
        + 0.3 * df["cfo_quality"].fillna(0)
        + 0.15 * df["current_ratio"].fillna(0)
        - df["leverage"].fillna(0)
    )

    # ----- 微观主信号：稳健轨 + 反转轨（对齐低波 v3 的高 IC_IR 结构）-----
    # 过热越高 → heat 越大 → 最终取反
    heat_cols = {
        "vamp_dev": 0.22,
        "vwap_dev": 0.14,
        "avgpx_dev": 0.10,
        "pressure_drift": 0.16,
        "obi_mean": 0.14,
        "depth3_mean": 0.08,
        "slope_asym": 0.08,
        "spoof_gap": 0.08,
    }
    # 不稳健越高 → unstable 越大 → 最终取反
    unstable_cols = {
        "obi_instability": 0.28,
        "amihud": 0.22,
        "rel_spread": 0.18,
        "avg_range": 0.14,
        "vol_cv": 0.10,
        "fragmentation": 0.08,
    }

    for col in list(heat_cols) + list(unstable_cols) + ["fin_quality", "day_range"]:
        if col in df.columns:
            df[col] = df.groupby("date")[col].transform(
                lambda s: s.fillna(s.median()) if s.notna().any() else s
            )

    heat = np.zeros(len(df))
    for col, w in heat_cols.items():
        heat += w * df.groupby("date")[col].rank(pct=True, method="average").to_numpy()

    unstable = np.zeros(len(df))
    for col, w in unstable_cols.items():
        unstable += w * df.groupby("date")[col].rank(pct=True, method="average").to_numpy()

    # 微观 alpha：低过热 + 高稳健（越大越好）
    df["micro_alpha"] = 0.52 * (1.0 - heat) + 0.48 * (1.0 - unstable)

    # 质量软门控：避免直接把 ROE 等拥挤信号塞进 Elastic Net
    q_pct = df.groupby("date")["fin_quality"].rank(pct=True, method="average")
    df["q_gate"] = 0.55 + 0.90 * q_pct
    df["gated_micro"] = df["micro_alpha"] * df["q_gate"]

    # 轻度行业相对确认（exposure 内合成，再正交掉）
    ind_keys = ["date", "industry_level1_code"]
    style_raw = ["SIZE", "BETA", "MOMENTUM", "RESVOL", "SIZENL",
                 "BTOP", "LIQUIDTY", "EARNYILD", "GROWTH", "LEVERAGE"]
    for col in style_raw:
        df[f"z_{col}"] = df.groupby(ind_keys)[col].transform(
            lambda s: _z(s.fillna(s.median()) if s.notna().any() else s.fillna(0))
        )
    df["style_qv_light"] = (
        0.30 * df["z_EARNYILD"]
        + 0.25 * df["z_BTOP"]
        + 0.15 * df["z_GROWTH"]
        - 0.15 * df["z_LEVERAGE"]
        - 0.10 * df["z_RESVOL"]
        - 0.05 * df["z_SIZENL"]
    )

    # ----- 控制变量：BARRA + 池内常见列（抬升 ModelScore 增量）-----
    df["inv_pe"] = -np.log1p(df["pe_ttm"].clip(lower=0))
    df["inv_pb"] = -np.log1p(df["pb"].clip(lower=0))
    df["log_mcap"] = np.log(df["float_market_cap"].where(df["float_market_cap"] > 0))

    control = [
        "z_SIZE", "z_BETA", "z_MOMENTUM", "z_LIQUIDTY", "z_RESVOL",
        "inv_pe", "inv_pb", "turn", "netflow_amount_rate_main", "roe_avg_ttm",
    ]
    for c in control:
        df[c] = df.groupby("date")[c].transform(
            lambda s: s.fillna(s.median()) if s.notna().any() else s.fillna(0)
        )
        df[c] = df[c].fillna(0.0)

    for col in ["micro_alpha", "gated_micro", "style_qv_light"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df.groupby("date")[col].transform(
            lambda s: s.fillna(s.median()) if s.notna().any() else s
        )
        df[f"r_{col}"] = _ols_resid_by_date(df, col, control)

    # 合成：微观残差主导（B 项），轻度风格残差仅作确认
    weights = {
        "r_gated_micro": 0.62,
        "r_micro_alpha": 0.28,
        "r_style_qv_light": 0.10,
    }
    for col in weights:
        df[col] = df.groupby("date")[col].transform(
            lambda s: s.fillna(s.median()) if s.notna().any() else s
        )

    df["factor"] = 0.0
    for col, w in weights.items():
        pct = df.groupby("date")[col].rank(pct=True, method="average")
        df["factor"] += w * pct

    df = df[
        (df["date"] >= pd.to_datetime(start_date))
        & (df["date"] <= pd.to_datetime(end_date))
    ]
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])

    out = df[["date", "instrument", "factor"]].drop_duplicates(
        ["date", "instrument"], keep="last"
    )
    out["factor"] = out["factor"].astype("float64")
    return out.sort_values(["date", "instrument"]).reset_index(drop=True)
